 ##  NoSQL Tutorial using MongoDB


In this tutorial we will go over some basic MongoDB queries and perform Machine Learning (ML) task using a NoSQL database

### MongoDB

MongoDB is a highly popular NoSQL database management system known for its document-oriented approach. Unlike traditional relational databases, MongoDB stores data in JSON-like documents. One of MongoDB's key features is its schema-less nature, allowing for the storage of heterogeneous data within the same collection without requiring a predefined schema. 

In [1]:
import pymongo
import pandas as pd
import json
import numpy as np

#### Connecting to the MongoDB Instance

In [2]:
# Connect to MongoDB using the IP address of the container
client = pymongo.MongoClient('mongodb://my-mongo-container:27017')

#### Initializing the Database

In [3]:
db = client['spotify_db']

#### Loading the Dataset
The Spotify Artist Data dataset contains information about albums of various artists across the United States. 


In [4]:
# Define the target collection
collection = db['data_artist']  # This is your target collection

# Path to the JSON file
json_file_path = '../data/data_artists.json'

# Load JSON data from the file
with open(json_file_path, 'r') as f:
    data = json.load(f)

# Flatten the structure into a list of albums, preserving artist information
albums_to_insert = []
for artist_key, artist_info in data.items():
    for album in artist_info[0]['albums_full']:  # Assuming each artist_key contains a list with one artist info
        # Include artist information within each album document
        album['artist_name'] = artist_info[0]['artist_name']
        album['artist_id'] = artist_info[0]['artist_id']
        albums_to_insert.append(album)

# Insert albums into the collection in smaller batches to manage memory and size limitations
def insert_in_batches(collection, documents, batch_size=100):
    total_inserted = 0
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        result = collection.insert_many(batch)
        total_inserted += len(result.inserted_ids)
    return total_inserted

# Call the function to insert data in batches
total_inserted = insert_in_batches(collection, albums_to_insert, batch_size=100)  # Adjust batch_size based on your needs

# Print the result of the insertion
print(f"Inserted {total_inserted} albums into the collection.")


Inserted 1089 albums into the collection.


#### Collection Name
As we have seen in class (slide 21 in the slide deck "13 - nosql.pdf", in MongoDB, documents are stored in databases which contain Let's look at the *data_artist* collection


In [5]:
collection = db['data_artist'] 


#### Collections in MongoDB

In MongoDB, a collection is a group of documents. It is the equivalent of a table in a relational database management system (RDBMS) like MySQL or PostgreSQL. Collections do not enforce a schema, meaning the documents within a collection can have different fields and structures.

In [6]:
# Get the list of collection names
collection_names = db.list_collection_names()

# Print the list of collection names
for collection_name in collection_names:
    print(collection_name)

data_artist


#### Data Content

- Each document in the collection represents an album by an artist.
- The document contains information about the album, such as its name, popularity, release date, and a list of tracks.
- Each track within the album has details like its name, unique track ID, and features such as danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness, liveness, valence, and tempo.

## Query Documents


The MongoDB document query is a method used to retrieve specific documents from a MongoDB collection based on specified criteria. It allows users to filter data and retrieve only the documents that match certain conditions.

### Example

Suppose we have a MongoDB collection named "books" containing documents with fields like "title," "author," and "genre." We want to query all books written by a specific author.

db.books.find({ author: "J.K. Rowling" })

For reference you can look at the official MongoDB query document tutorial
https://www.mongodb.com/docs/manual/tutorial/query-documents/

#### Find one document
Find album names by the artist "ye"

In [7]:
document = collection.find_one({'album_name': 'ye'})
print(document)


{'_id': ObjectId('69c57238feb9bcfe1aa4ed50'), 'album_name': 'ye', 'album_popularity': 78, 'release_date': '2018-06-01', 'tracks': [{'track_name': 'I Thought About Killing You', 'track_id': '0yhxBvedRdGxsPZHJNI4VA', 'features': [{'danceability': 0.752, 'energy': 0.396, 'key': 2, 'loudness': -7.932, 'mode': 1, 'speechiness': 0.411, 'acousticness': 0.867, 'instrumentalness': 0, 'liveness': 0.0486, 'valence': 0.417, 'tempo': 116.016}]}, {'track_name': 'Yikes', 'track_id': '1qsHYUd2c1wFGcn7e63QmG', 'features': [{'danceability': 0.683, 'energy': 0.776, 'key': 1, 'loudness': -3.251, 'mode': 0, 'speechiness': 0.341, 'acousticness': 0.15, 'instrumentalness': 0, 'liveness': 0.227, 'valence': 0.223, 'tempo': 173.996}]}, {'track_name': 'All Mine', 'track_id': '3U21A07gAloCc4P7J8rxcn', 'features': [{'danceability': 0.925, 'energy': 0.31, 'key': 11, 'loudness': -6.531, 'mode': 0, 'speechiness': 0.291, 'acousticness': 0.123, 'instrumentalness': 0, 'liveness': 0.0931, 'valence': 0.291, 'tempo': 121.92

#### Find documents based on a condition
We want to find albums whose populated is greater than 95

In [8]:
documents = collection.find({'album_popularity': {'$gt': 95}})
for doc in documents:
    print(doc)

{'_id': ObjectId('69c57238feb9bcfe1aa4ee8e'), 'album_name': 'Fine Line', 'album_popularity': 97, 'release_date': '2019-12-13', 'tracks': [{'track_name': 'Golden', 'track_id': '45S5WTQEGOB1VHr1Q4FuPl', 'features': [{'danceability': 0.448, 'energy': 0.838, 'key': 4, 'loudness': -5.257, 'mode': 0, 'speechiness': 0.0557, 'acousticness': 0.21, 'instrumentalness': 0.000131, 'liveness': 0.131, 'valence': 0.254, 'tempo': 139.863}]}, {'track_name': 'Watermelon Sugar', 'track_id': '6UelLqGlWMcVH1E5c4H7lY', 'features': [{'danceability': 0.548, 'energy': 0.816, 'key': 0, 'loudness': -4.209, 'mode': 1, 'speechiness': 0.0465, 'acousticness': 0.122, 'instrumentalness': 0, 'liveness': 0.335, 'valence': 0.557, 'tempo': 95.39}]}, {'track_name': 'Adore You', 'track_id': '3jjujdWJ72nww5eGnfs2E7', 'features': [{'danceability': 0.676, 'energy': 0.771, 'key': 8, 'loudness': -3.675, 'mode': 1, 'speechiness': 0.0483, 'acousticness': 0.0237, 'instrumentalness': 7e-06, 'liveness': 0.102, 'valence': 0.569, 'tempo

## Aggregation Pipelines

Aggregation pipelines are used to perform aggregation queries in MongoDB. An aggregation pipeline consists of one or more stages that process documents:

- Each stage performs an operation on the input documents. For example, a stage can filter documents, group documents, and calculate values.
  
- The documents that are output from a stage are passed to the next stage.

- For reference: https://www.mongodb.com/docs/manual/aggregation/

Refer to the comments in the code below to understand what each statement does

In [9]:
from pymongo import MongoClient

# Example aggregation pipeline
pipeline = [
    {"$match": {"album_popularity": {"$gt": 80}}},  # Filter albums with popularity greater than 80
    {"$group": {"_id": "$artist_name", "total_albums": {"$sum": 1}}},  # Group albums by artist and count total albums
    {"$sort": {"total_albums": -1}},  # Sort artists by total albums in descending order
    {"$limit": 5}  # Limit the result to top 5 artists
]

# Execute the aggregation pipeline
result = collection.aggregate(pipeline)

# Print the result
for doc in result:
    print(doc)


{'_id': 'One Direction', 'total_albums': 5}
{'_id': 'Taylor Swift', 'total_albums': 3}
{'_id': 'The Beatles', 'total_albums': 2}
{'_id': 'Coldplay', 'total_albums': 2}
{'_id': 'Khalid', 'total_albums': 2}


In [10]:
# Example aggregation pipeline
pipeline = [
    {"$match": {"album_name": "JESUS IS KING"}}, # Match documents where the album_name is "JESUS IS KING"
    {"$unwind": "$tracks"}, # Unwind the tracks array into individual documents for each track. This stage is used here to transform each document with an array of tracks into separate documents for each track in the array. This is necessary for the subsequent grouping operation if the tracks field is an array of track details.
    {"$group": {"_id": "$album_name", "total_tracks": {"$sum": 1}}}, # Group the documents by album_name and count the total number of tracks per album
    {"$project": {"_id": 0, "album_name": "$_id", "total_tracks": 1}} # Project (or reshape) the output documents to include album_name and total_tracks fields
]

# Execute the aggregation pipeline
result = collection.aggregate(pipeline)

# Print the result
for doc in result:
    print(doc)


{'total_tracks': 11, 'album_name': 'JESUS IS KING'}


The following code represents a MongoDB aggregation pipeline designed to analyze and summarize the musical characteristics of tracks within documents in a collection. Initially, it flattens the nested arrays tracks and tracks.features by unwinding them, which allows for per-track analysis rather than treating the tracks array as a single document field. Subsequently, it groups the documents by their MongoDB default _id, preserving the uniqueness of each document while calculating various averages of musical features such as danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness, liveness, valence, and tempo. These averages are computed across all tracks and features within each original document.

After aggregating these statistics, the pipeline then re-integrates this summarized data back into the original document structure. It does so by initially retaining the entire original document (before aggregation) as a field named first, and then adding the calculated average values to this first document. The final stage of the pipeline replaces the original document root with this modified first document, effectively updating each document with its corresponding average musical feature values. This approach provides a comprehensive and detailed analysis of the musical content within each document, augmenting the original data with valuable insights derived from the aggregation process.

In [11]:
# Pipeline to aggregate data
pipeline = [
    # Unwind the 'tracks' array to flatten it
    {"$unwind": "$tracks"},
    # Unwind the 'features' array within each track to flatten it
    {"$unwind": "$tracks.features"},
    # Group documents by their default document ID (i.e., '_id')
    {"$group": {
        "_id": "$_id",  # Use MongoDB's default document ID as the unique identifier
        # Calculate the average danceability for each document
        "avg_danceability": {"$avg": "$tracks.features.danceability"},
        # Calculate the average energy for each document
        "avg_energy": {"$avg": "$tracks.features.energy"},
        # Calculate the average key for each document
        "avg_key": {"$avg": "$tracks.features.key"},
        # Calculate the average loudness for each document
        "avg_loudness": {"$avg": "$tracks.features.loudness"},
        # Calculate the average mode for each document
        "avg_mode": {"$avg": "$tracks.features.mode"},
        # Calculate the average speechiness for each document
        "avg_speechiness": {"$avg": "$tracks.features.speechiness"},
        # Calculate the average acousticness for each document
        "avg_acousticness": {"$avg": "$tracks.features.acousticness"},
        # Calculate the average instrumentalness for each document
        "avg_instrumentalness": {"$avg": "$tracks.features.instrumentalness"},
        # Calculate the average liveness for each document
        "avg_liveness": {"$avg": "$tracks.features.liveness"},
        # Calculate the average valence for each document
        "avg_valence": {"$avg": "$tracks.features.valence"},
        # Calculate the average tempo for each document
        "avg_tempo": {"$avg": "$tracks.features.tempo"},
        # Retain the entire original document as 'first'
        "first": {"$first": "$$ROOT"}
    }},
    # Add newly calculated fields to the original document
    {"$addFields": {
        "first.avg_danceability": "$avg_danceability",
        "first.avg_energy": "$avg_energy",
        "first.avg_key": "$avg_key",
        "first.avg_loudness": "$avg_loudness",
        "first.avg_mode": "$avg_mode",
        "first.avg_speechiness": "$avg_speechiness",
        "first.avg_acousticness": "$avg_acousticness",
        "first.avg_instrumentalness": "$avg_instrumentalness",
        "first.avg_liveness": "$avg_liveness",
        "first.avg_valence": "$avg_valence",
        "first.avg_tempo": "$avg_tempo"
    }},
    # Replace the root document with the modified 'first' document
    {"$replaceRoot": {"newRoot": "$first"}}
]

# Execute the aggregation pipeline and convert the result to a list
result = list(collection.aggregate(pipeline))

# Create a DataFrame from the aggregated result
df_albums = pd.DataFrame(result)

# Now 'df_albums' contains the aggregated data in a pandas DataFrame with additional features


In [12]:
# Display DataFrame
df_albums.head()

,_id,album_name,album_popularity,release_date,tracks,artist_name,artist_id,avg_danceability,avg_energy,avg_key,avg_loudness,avg_mode,avg_speechiness,avg_acousticness,avg_instrumentalness,avg_liveness,avg_valence,avg_tempo
0,69c57238feb9bcfe1aa4ef3b,ライヴ・イン・コネチカット1975 (Live),4,2018-10-24,"{'track_name': 'マンデイ・モーニング (Live)', 'track_id'...",Fleetwood Mac,08GQAI4eElDnROBrJRGE0X,0.360889,0.703444,4.222222,-8.650222,0.777778,0.047311,0.104733,0.044665,0.648222,0.478444,125.989111
1,69c57238feb9bcfe1aa4f0e4,One of These Nights (2013 Remaster),71,1975,{'track_name': 'One of These Nights - 2013 Rem...,Eagles,0ECwFtbIWEVNwjlrfc6xoL,0.496889,0.536000,6.000000,-9.651889,0.888889,0.030633,0.227600,0.146775,0.138178,0.525578,113.467556
2,69c57238feb9bcfe1aa4f16f,Binaural,50,2000-05-16,"{'track_name': 'Breakerfall', 'track_id': '669...",Pearl Jam,1w5Kfo2jwwIPruYS2UWh56,0.411692,0.640519,5.846154,-9.042846,0.846154,0.054269,0.206018,0.245896,0.214300,0.489438,145.829231
3,69c57238feb9bcfe1aa4ef01,Never Let Me Down [(Remaster) [Japanese Version]],34,1987-04-20,{'track_name': 'Day-In Day-Out - 2018 Remaster...,David Bowie,0oSGxfWSnnOXhD2fKuz2Gy,0.615636,0.907091,5.363636,-6.699818,0.636364,0.042545,0.074233,0.013242,0.169082,0.631818,126.588455
4,69c57238feb9bcfe1aa4eed8,Madman Across The Water,74,1971-11-05,"{'track_name': 'Tiny Dancer', 'track_id': '2TV...",Elton John,3PhoLpVuITZKcymswpck5b,0.395889,0.469444,4.444444,-12.614556,0.888889,0.036311,0.428011,0.003297,0.153422,0.409444,142.108111


### Linear Regression for predicting Album popularity
Now, for the ML part, we want to predict the Album popularity using other features such as avg_danceability, avg_energy, etc. We will train a linear regression model to predict those values

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Extract features and target variable
X = df_albums[['avg_danceability', 'avg_energy', 'avg_key', 'avg_loudness', 'avg_mode',
               'avg_speechiness', 'avg_acousticness', 'avg_instrumentalness',
               'avg_liveness', 'avg_valence', 'avg_tempo']]
y = df_albums['album_popularity']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict album popularity on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)

# Calculate root mean squared error
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error: {rmse}")



Root Mean Squared Error: 20.156938224255146


### Recommendation of Songs using Likes and dislikes as label columns 

We will use 2 datasets (good.json and dislike.json) to construct a (merged) Pandas dataframe that contains tracks that people like and dislike. We will then create a label which takes in "like" or "dislike" values to distinguish between them. Finally, we will use a classifier to classify songs into one of the two classes (like, dislike). For information about how KNN classifiers internally work, refer to: https://mkang.faculty.unlv.edu/teaching/CS489_689/05.KNN.pdf

#### Loading the dataset

In [14]:
import pymongo
import pandas as pd


# Define collection names
likes_collection = db["likes"]
dislikes_collection = db["dislikes"]

# Load JSON data
with open("../data/good.json", "r") as likes_file:
    likes_data = json.load(likes_file)

with open("../data/dislike.json", "r") as dislikes_file:
    dislikes_data = json.load(dislikes_file)

# Insert JSON data into MongoDB collections
likes_collection.insert_many(likes_data["audio_features"])
dislikes_collection.insert_many(dislikes_data["audio_features"])

# Retrieve data from MongoDB collections
likes_df = pd.DataFrame(list(likes_collection.find()))
dislikes_df = pd.DataFrame(list(dislikes_collection.find()))


# Add label column to the likes and dislikes DataFrames
likes_df['label'] = 'like'
dislikes_df['label'] = 'dislike'

# Concatenate the likes and dislikes DataFrames
merged_df = pd.concat([likes_df, dislikes_df], ignore_index=True)


In [15]:
likes_df.columns

Index(['_id', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'type', 'id', 'uri', 'track_href', 'analysis_url',
       'duration_ms', 'time_signature', 'label'],
      dtype='object')

In [16]:
merged_df.head()

,_id,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,label
0,69c57239feb9bcfe1aa4f18f,0.749,0.839,6,-4.847,1,0.297,0.0867,0.000000,0.2040,0.804,172.068,audio_features,55mcupbf7cIsuCEVAuTJVk,spotify:track:55mcupbf7cIsuCEVAuTJVk,https://api.spotify.com/v1/tracks/55mcupbf7cIs...,https://api.spotify.com/v1/audio-analysis/55mc...,111000,4,like
1,69c57239feb9bcfe1aa4f190,0.573,0.581,10,-9.026,0,0.339,0.7530,0.000001,0.1300,0.351,76.506,audio_features,57RtLWT7IpugV0yi5bsxJk,spotify:track:57RtLWT7IpugV0yi5bsxJk,https://api.spotify.com/v1/tracks/57RtLWT7Ipug...,https://api.spotify.com/v1/audio-analysis/57Rt...,169347,4,like
2,69c57239feb9bcfe1aa4f191,0.800,0.719,7,-6.262,1,0.234,0.1090,0.000000,0.0580,0.815,143.975,audio_features,5VyfAfp2Yt3qaeuvq55ll3,spotify:track:5VyfAfp2Yt3qaeuvq55ll3,https://api.spotify.com/v1/tracks/5VyfAfp2Yt3q...,https://api.spotify.com/v1/audio-analysis/5Vyf...,230854,4,like
3,69c57239feb9bcfe1aa4f192,0.778,0.632,8,-6.415,1,0.125,0.0404,0.000000,0.0912,0.827,140.951,audio_features,3eWHY75nDgte70hh5yf4UW,spotify:track:3eWHY75nDgte70hh5yf4UW,https://api.spotify.com/v1/tracks/3eWHY75nDgte...,https://api.spotify.com/v1/audio-analysis/3eWH...,224029,4,like
4,69c57239feb9bcfe1aa4f193,0.797,0.852,8,-5.202,1,0.241,0.0555,0.000024,0.0536,0.480,136.035,audio_features,2UwrB6Ge6mPfUV8yGvAfX7,spotify:track:2UwrB6Ge6mPfUV8yGvAfX7,https://api.spotify.com/v1/tracks/2UwrB6Ge6mPf...,https://api.spotify.com/v1/audio-analysis/2Uwr...,102353,4,like


After creating the merged dataframe, we are now ready to train the classifier

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Drop the specified columns
columns_to_drop = ['type', 'uri', 'track_href', 'analysis_url','_id','id']
df_filtered = merged_df.drop(columns=columns_to_drop)

# Select only integer columns for training
X = df_filtered.drop(columns=['label'])

# Extract the target variable
y = df_filtered['label']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Instantiate the kNN Classifier
k = 5  # Set the value of k
knn_classifier = KNeighborsClassifier(n_neighbors=k)

# Train the model
knn_classifier.fit(X_train_scaled, y_train)

# Make predictions
y_pred = knn_classifier.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")


Accuracy: 0.9230769230769231
